In [2]:
#DENSE_RANK or NTILE ka practical use.

In [3]:
#1.DENSE_RANK() - RANK() jaisa hi hai, lekin ties ke baad rank skip nahi karta (1,1,2,3...instead of 1,1,2,3,4...).

In [4]:
#2.NTILE(n) - rows ko n barabar groups/buckets me baant deta hai(jaise NTILE(4) se quartiles ban jaate hai - customer segmentation ke liye bahut kaam aata hai).

In [5]:
#01:Teeno ranking functions side-by-side compare karna(fark clearly samjhne ke liye):

In [6]:
import pandas as pd 
import sqlite3

conn = sqlite3.connect('../data/db/ecommerce.db')

q1 = pd.read_sql("""
    SELECT "Customer ID", total_spend,
            ROW_NUMBER() OVER (ORDER BY total_spend DESC) as row_num,
            RANK() OVER (ORDER BY total_spend DESC) as rank_val,
            DENSE_RANK() OVER (ORDER BY total_spend DESC) as dense_rank_val
    FROM (
        SELECT "Customer ID", SUM(OrderValue) as total_spend
        FROM orders
        GROUP BY "Customer ID"
    )
    ORDER BY total_spend DESC
    LIMIT 15
""",conn)
print(q1)        


    Customer ID  total_spend  row_num  rank_val  dense_rank_val
0       18102.0    598215.22        1         1               1
1       14646.0    523342.07        2         2               2
2       14156.0    296564.69        3         3               3
3       14911.0    270248.53        4         4               4
4       17450.0    233579.39        5         5               5
5       13694.0    190825.52        6         6               6
6       17511.0    171885.98        7         7               7
7       12415.0    143269.29        8         8               8
8       16684.0    141502.25        9         9               9
9       15061.0    136391.48       10        10              10
10      15311.0    113513.07       11        11              11
11      13089.0    113214.19       12        12              12
12      17949.0     98895.59       13        13              13
13      16029.0     91800.91       14        14              14
14      14298.0     90489.31       15   

In [7]:
#jaha ties hongi(same total_spend), waha teeno columns ka fark clearly dikhega.

In [8]:
#02:NTILE(4) - customers ko 4 quartiles me baatna (spending ke hisaab se):

In [10]:
q2 = pd.read_sql("""
    SELECT "Customer ID", total_spend,
            NTILE(4) OVER (ORDER BY total_spend DESC) as spend_quartile
    FROM (
        SELECT "Customer ID", SUM(OrderValue) as total_spend
        FROM orders
        GROUP BY "Customer ID"
    )
    ORDER BY total_spend DESC
""",conn)
print(q2.head(20))
print(q2['spend_quartile'].value_counts().sort_index())

    Customer ID  total_spend  spend_quartile
0       18102.0    598215.22               1
1       14646.0    523342.07               1
2       14156.0    296564.69               1
3       14911.0    270248.53               1
4       17450.0    233579.39               1
5       13694.0    190825.52               1
6       17511.0    171885.98               1
7       12415.0    143269.29               1
8       16684.0    141502.25               1
9       15061.0    136391.48               1
10      15311.0    113513.07               1
11      13089.0    113214.19               1
12      17949.0     98895.59               1
13      16029.0     91800.91               1
14      14298.0     90489.31               1
15      15769.0     84269.38               1
16      13798.0     73573.47               1
17      15838.0     73404.11               1
18      12931.0     71299.67               1
19      17841.0     69516.19               1
spend_quartile
1    1486
2    1486
3    1485
4    1485


In [11]:
#spend_quartile = 1 matlab top 25% spenders(highest value customers),4 matlab bottom 25%(sabse kam kharch karne wale)- yeh RFM/churn analysis me customer segmentation ke liye directly kaam aaiga(month 5, day 81 me).

In [12]:
#03:NTILE (10)- deciles banana(or bhi granular segmentation):

In [16]:
q3 = pd.read_sql("""
    SELECT "Customer ID", total_spend,
            NTILE(10) OVER (ORDER BY total_spend DESC) as spend_decile
    FROM (
        SELECT "Customer ID" , SUM(OrderValue) as total_spend
        FROM orders
        GROUP BY "Customer ID"
    )
""",conn)
print(q3['spend_decile'].value_counts().sort_index())

spend_decile
1     595
2     595
3     594
4     594
5     594
6     594
7     594
8     594
9     594
10    594
Name: count, dtype: int64


In [19]:
#04:DENSE_RANK()+PARTITION BY - har country ke andar dense ranking:

In [18]:
q4 = pd.read_sql("""
    SELECT c."Customer ID", c.Country, total_spend,
           DENSE_RANK() OVER (PARTITION BY c.Country ORDER BY total_spend DESC) as country_dense_rank
    FROM customers c
    INNER JOIN (
        SELECT "Customer ID", SUM(OrderValue) as total_spend
        FROM orders
        GROUP BY "Customer ID"
    ) o ON c."Customer ID" = o."Customer ID"
    ORDER BY c.Country, country_dense_rank
""", conn)
print(q4.head(20))

    Customer ID    Country  total_spend  country_dense_rank
0       12415.0  Australia    143269.29                   1
1       12431.0  Australia     10719.41                   2
2       12422.0  Australia      4119.35                   3
3       12388.0  Australia      3901.11                   4
4       12424.0  Australia      3289.78                   5
5       12393.0  Australia      2376.75                   6
6       12389.0  Australia      1433.33                   7
7       12434.0  Australia      1062.48                   8
8       12386.0  Australia       660.80                   9
9       12411.0  Australia       346.90                  10
10      16321.0  Australia       265.50                  11
11      12392.0  Australia       234.75                  12
12      12400.0  Australia       205.25                  13
13      12416.0  Australia       202.56                  14
14      12387.0  Australia       143.94                  15
15      12429.0    Austria      7435.51 

In [20]:
#practice questions.

In [23]:
#1.NTILE(4) se jo quartile 1 (top spenders) mile, unka average total_spend nikaalo aur quartile 4 (bottom) se compare karo — fark kitna bada hai?

In [22]:
#

In [24]:
#2.Ek query likho jo dikhaye ki kitne customers ke beech ties hain (RANK() aur DENSE_RANK() ka value same hai lekin ROW_NUMBER() alag) — yeh confirm karega ki tumhe fark samajh aaya.

In [25]:
#

In [26]:
conn.close()